# SafeRest360 — Pipeline de Entrenamiento

Este notebook ejecuta el pipeline completo de Transfer Learning para detección de personas:
1. Preparación del dataset (COCO 2017 → clase persona → formato YOLO)
2. Entrenamiento de 4 variantes YOLO
3. Evaluación y comparación

**Requisitos**: GPU runtime (T4 o superior)

In [ ]:
# Verificar GPU
!nvidia-smi
import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

In [ ]:
# Instalar dependencias
!pip install -q ultralytics albumentations supervision mlflow
# YOLO-NAS (opcional, Apache-2.0)
!pip install -q super-gradients

In [ ]:
# Opción A: Subir vision-engine desde Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copiar vision-engine desde Drive (sube la carpeta SafeRest360 a tu Drive primero)
!cp -r /content/drive/MyDrive/SafeRest360/vision-engine /content/vision-engine

# Opción B: Clonar desde GitHub (si ya tienes el repo)
# !git clone https://github.com/TU_USUARIO/SafeRest360.git
# !cp -r SafeRest360/vision-engine /content/vision-engine

import sys
sys.path.insert(0, '/content/vision-engine')

## 1. Descarga y preparación del dataset COCO 2017

In [ ]:
import os
os.makedirs('data', exist_ok=True)

# Descargar anotaciones COCO 2017
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O data/annotations.zip
!unzip -q data/annotations.zip -d data/

# Descargar imágenes (val2017 es más pequeño para pruebas rápidas)
!wget -q http://images.cocodataset.org/zips/val2017.zip -O data/val2017.zip
!unzip -q data/val2017.zip -d data/

# Para entrenamiento completo, descomentar:
# !wget -q http://images.cocodataset.org/zips/train2017.zip -O data/train2017.zip
# !unzip -q data/train2017.zip -d data/

In [ ]:
from pathlib import Path
from src.dataset.coco_extractor import extract_person_subset, create_dataset_yaml

# Extraer solo personas de COCO val2017
stats = extract_person_subset(
    annotation_path=Path('data/annotations/instances_val2017.json'),
    images_dir=Path('data/val2017'),
    output_dir=Path('data/person_dataset'),
    max_images=2000  # Limitar para demo rápida
)
print(f"Dataset: {stats}")

In [ ]:
from src.dataset.splitter import split_dataset

# Split train/val/test
split_stats = split_dataset(
    source_dir=Path('data/person_dataset'),
    output_dir=Path('data/person_yolo'),
    train_ratio=0.7,
    val_ratio=0.2,
    test_ratio=0.1,
    seed=42
)
print(f"Split: {split_stats}")

# Crear dataset.yaml
yaml_path = create_dataset_yaml(Path('data/person_yolo'))
print(f"Dataset config: {yaml_path}")

In [ ]:
from src.dataset.augmentation import augment_dataset

# Aumentar datos de entrenamiento
aug_stats = augment_dataset(
    images_dir=Path('data/person_yolo/train/images'),
    labels_dir=Path('data/person_yolo/train/labels'),
    output_images_dir=Path('data/person_yolo/train/images'),
    output_labels_dir=Path('data/person_yolo/train/labels'),
    augmentations_per_image=2
)
print(f"Augmentation: {aug_stats}")

## 2. Entrenamiento — Transfer Learning

Entrenamos 4 variantes con pesos preentrenados en COCO:

| Modelo | Licencia | Parámetros |
|--------|----------|------------|
| YOLOv8n | AGPL-3.0 | 3.2M |
| YOLOv8s | AGPL-3.0 | 11.2M |
| YOLOv8m | AGPL-3.0 | 25.9M |
| YOLO-NAS-S | Apache-2.0 | ~12M |

In [ ]:
from src.training.trainer import run_experiment
from src.training.config import TrainingConfig, VARIANT_CONFIGS

dataset_yaml = str(Path('data/person_yolo/dataset.yaml'))
results = {}

for variant_name, variant_cfg in VARIANT_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"Entrenando: {variant_name}")
    print(f"{'='*60}")
    
    config = TrainingConfig(
        model_variant=variant_cfg['model_variant'],
        pretrained_weights=variant_cfg['pretrained_weights'],
        epochs=100,
        imgsz=640,
        batch=16,
        patience=20,
    )
    
    record = run_experiment(
        config=config,
        dataset_yaml=dataset_yaml,
        output_dir=Path(f'runs/{variant_name}'),
    )
    results[variant_name] = record
    print(f"Completado: {variant_name} — mAP50: {record.get('metrics', {}).get('mAP50', 'N/A')}")

## 3. Evaluación y comparación

In [ ]:
from src.evaluation.compare import build_comparison_table, format_comparison_markdown

# Cargar todos los resultados
experiment_dirs = [Path(f'runs/{v}') for v in VARIANT_CONFIGS.keys()]
table = build_comparison_table(experiment_dirs)
print(format_comparison_markdown(table))

In [ ]:
from src.evaluation.benchmark import benchmark_model

# Benchmark de latencia
for variant_name in VARIANT_CONFIGS.keys():
    best_pt = Path(f'runs/{variant_name}/weights/best.pt')
    if best_pt.exists():
        bench = benchmark_model(str(best_pt), imgsz=640)
        print(f"{variant_name}: {bench['mean_latency_ms']:.1f}ms, {bench['fps']:.1f} FPS")

In [ ]:
# Visualizar comparación
import matplotlib.pyplot as plt
import numpy as np

if table:
    names = [r['variant'] for r in table]
    map50 = [r.get('mAP50', 0) for r in table]
    fps = [r.get('fps', 0) for r in table]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    ax1.barh(names, map50, color=['#3b82f6', '#10b981', '#f59e0b', '#ef4444'])
    ax1.set_xlabel('mAP@50')
    ax1.set_title('Precisión (mAP@50)')
    
    ax2.barh(names, fps, color=['#3b82f6', '#10b981', '#f59e0b', '#ef4444'])
    ax2.set_xlabel('FPS')
    ax2.set_title('Velocidad (FPS)')
    
    plt.tight_layout()
    plt.savefig('comparison.png', dpi=150)
    plt.show()
    print("Gráfico guardado en comparison.png")

## 4. Selección y exportación del mejor modelo

Criterios de selección:
- **Licencia**: YOLO-NAS-S (Apache-2.0) es preferible para SaaS comercial
- **Precisión**: mAP@50 > 0.80 es aceptable para vigilancia
- **Velocidad**: > 15 FPS requerido para procesamiento en tiempo real

In [ ]:
import shutil

# Seleccionar el mejor modelo (ajustar según resultados reales)
best_variant = 'yolov8s'  # Cambiar según los resultados
best_model_path = Path(f'runs/{best_variant}/weights/best.pt')

if best_model_path.exists():
    export_dir = Path('exported_models')
    export_dir.mkdir(exist_ok=True)
    shutil.copy2(best_model_path, export_dir / f'saferest360_{best_variant}_v1.pt')
    print(f"Modelo exportado: exported_models/saferest360_{best_variant}_v1.pt")
else:
    print(f"Modelo no encontrado: {best_model_path}")
    print("Asegúrate de ejecutar las celdas de entrenamiento primero.")